In [18]:
homedir = '/mnt/mirabelle/az6922_homedir/DRing/src/emp/datacentre/'
import random
import numpy as np

# from makec2s.ipynb
def genflowbytes():
    np.random.seed(0)
    
    mean_bytes = 100.0 * 1024
    shape = 1.05
    scale = mean_bytes * (shape - 1)/shape

    x = np.random.exponential(scale=1.0/shape)
    flowbytes = int(scale * np.exp(x))
    return flowbytes

def adjustbytesbymtu(flowbytes):
  mss = 1500
  return mss * ((flowbytes+mss-1)//mss)

large_flow_threshold = 10 * 1024 * 1024

In [22]:
stime = 144 # ms
nlinks = 2048
nhosts = 3072
bw = 1342176000 # B per second
# load_list = range(1,11) #[10,30,50,70]
# seed_list = [1,2,3,4,5]
topologytype = 1
nswitches = 80
os = 1
k = 64
nintervals = 1
npfile = 'evalnetpathfiles/netpath_leafspine_80_64_ecmp.np'
pwfile = 'experiments/nsdi26fall/eval_failure_tor/prv1/pwfiles/pathweight_leafspine.pw'
failpct_list = range(2,11,2)
load = 4
seed = 1
failseed_list = range(10)

(current dir: ~/DRing/src/emp/datacentre/experiments/nsdi26fall/)
cp eval_main/unv1/pwfiles/pathweight_leafspine.pw eval_failure_tor/prv1/pwfiles/pathweight_leafspine.pw

In [23]:
# generate connection_matrices file (1)
unv1bytes = 0
unv1file = f'{homedir}rawtrafficfiles/prv1'
maxinterval = 0
with open(unv1file, 'r') as f:
    lines = f.readlines()
    for line in lines:
        tokens = line.split(',')
        # 0,32,31,10500
        # interval,fromserver,toserver,bytes
        unv1bytes += int(tokens[3])
        maxinterval = max(maxinterval, int(tokens[0]))
print(f'unv1bytes {unv1bytes}, maxinterval {maxinterval}, fullload {bw * stime * nlinks / 1000}, ratio {(bw * stime * nlinks / 1000) / unv1bytes}')

unv1bytes 222240244500, maxinterval 7, fullload 395823808512.0, ratio 1.7810626936742773


In [24]:
# generate connection_matrices file (2)
random.seed(0)
totalbytes = bw * stime / 1000 * nlinks * load / 100  # B

os = 1
ls_lsx = int(3*k/4)
def server_to_sw(server):
    return int(server/os/ls_lsx)

for failpct in failpct_list:
    numfailtor = int(nswitches * failpct / 100)
    for failseed in failseed_list:
        torfailurefile = f"{homedir}/experiments/nsdi26fall/eval_failure_tor/torfailurefiles/leafspine_{numfailtor}_{failseed}.lf"
        torlist = list()
        with open(torfailurefile, 'r') as f:
            lines = f.readlines()
            for line in lines:
                torlist.append(int(line))

        cmfile = f'cmfiles/leafspine_load{load}_{numfailtor}_{failseed}.cm'
        mult = totalbytes / unv1bytes
        actualbytes = 0
        with open(cmfile, 'w') as fw:
            with open(unv1file, 'r') as fr:
                lines = fr.readlines()
                iline = 0
                while actualbytes < totalbytes:
                    line = lines[iline]
                    tokens = line.split(',')
                    interval = int(tokens[0])
                    fromserver = int(tokens[1])
                    toserver = int(tokens[2])
                    multbytes = int(tokens[3])

                    if fromserver >= nhosts or toserver >= nhosts:
                        iline += 1
                        if iline >= len(lines):
                            iline = 0
                            if mult-1>0:
                                mult = mult-1
                        continue

                    if mult >= 1 or (random.random() < mult):
                        multbytes = adjustbytesbymtu(multbytes)
        
                        # generate random start time
                        start_time_ms = random.uniform(0, stime//(maxinterval+1)) + interval * (stime//(maxinterval+1))

                        actualbytes += int(multbytes)

                        fromsw = server_to_sw(fromserver)
                        tosw = server_to_sw(toserver)
                        # print(f"fromserver {fromserver}, toserver {toserver}, fromsw {fromsw}, tosw {tosw}, multbytes {multbytes}, start_time_ms {start_time_ms:.4f}", end='\r')
                        if fromsw not in torlist and tosw not in torlist:
                            fw.write(f'{fromserver},{toserver},{int(multbytes)},{start_time_ms:.4f}\n')

                    iline += 1
                    if iline >= len(lines):
                        iline = 0
                        if mult-1>0:
                            mult = mult-1

                        # print(f'actualbytes {actualbytes}, totalbytes {totalbytes}, mult {mult}', end='\r')

        # print(f'load {load}%, totalbytes {totalbytes}, unv1bytes {unv1bytes}, mult {mult}, actualbytes {actualbytes}')


In [ ]:
# # generate linkfailurefiles
# with open(f"{homedir}experiments/nsdi26fall/eval_failure_link/leafspine_lffiles.conf", 'w') as f:
#     for failpct in failpct_list:
#         numfaillinks = int(nlinks * failpct / 100)
#         for failseed in failseed_list:
#             linkfailurefile = f"{homedir}/experiments/nsdi26fall/eval_failure_link/linkfailurefiles/leafspine_{numfaillinks}_{failseed}.lf"
#             f.write(f"python3 {homedir}generate_leafspine_linkfailurefiles.py --numfaillinks {numfaillinks} --rseed {failseed} --linkfailurefile {linkfailurefile}\n")

~~(current dir: ~/DRing/src/emp/datacenter/experiments/nsdi26fall/eval_failure_link/)
python3 ../../../pararun.py --conf leafspine_lffiles.conf --worker 20~~

In [25]:
# generate conf file
conffile = f'{homedir}experiments/nsdi26fall/eval_failure_tor/prv1/run_ls_before.conf'
with open(conffile, 'w') as f:
    for failpct in failpct_list:
        numfailtor = int(nswitches * failpct / 100)
        for failseed in failseed_list:
            cmfile = f'experiments/nsdi26fall/eval_failure_tor/prv1/cmfiles/leafspine_load{load}_{numfailtor}_{failseed}.cm'
            outfile = f'experiments/nsdi26fall/eval_failure_tor/prv1/outfiles/leafspine_before_nfailtor{numfailtor}_fseed{failseed}.out'
            logoutfile = f'experiments/nsdi26fall/eval_failure_tor/prv1/outfiles/leafspine_before_nfailtor{numfailtor}_fseed{failseed}.log'
            f.write(f"./eval -stime {stime} -seed {seed} -cmfile {cmfile} -topologytype {topologytype} -numswitches {nswitches} -numhosts {nhosts} -os {os} -ls_k {k} -npfile {npfile} -pwfileprefix {pwfile} -numintervals {nintervals} -o {logoutfile} > {outfile}\n")


python3 pararun.py --conf experiments/nsdi26fall/eval_failure_tor/prv1/run_ls_before.conf --worker 50